# Data Pipeline

Builds the stratified dataset used to train the binary biology refusal classifier.

**Positives** (label = 1): WMDP-bio question stems (answer choices stripped), plus programmatic prose rephrasings to test format generalization.

**Negatives** (label = 0), stratified across four buckets to control surface-feature confounds:

- `mmlu_bio` — benign biology MCQs (controls against "biology → refuse")
- `mmlu_other` — non-bio technical MCQs (controls against "MCQ format → refuse")
- `alpaca` — general instructions (baseline traffic distribution)
- `dual_use` — virology / genetics / anatomy + filtered PubMedQA (adjacency negatives)

**Splits**: topic-clustered split on WMDP positives (KMeans, k=30, rebalanced by cumulative cluster size to hit ~20% test). Per-bucket stratified 80/20 split on negatives.

Runs end-to-end on Colab free tier, CPU. ~5–10 minutes.

## Environment setup

The cells below mount Google Drive and install dependencies. All outputs persist to `/content/drive/MyDrive/biology_refusal/data/`.

In [ ]:
!pip install -q datasets sentence-transformers scikit-learn pandas pyarrow

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

In [ ]:

import re
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = Path("./data")
OUT_DIR.mkdir(exist_ok=True)

## Load WMDP-bio positives

In [ ]:
# Keep only the question stem.

print("Loading WMDP-bio...")
wmdp = load_dataset("cais/wmdp", "wmdp-bio", split="test")
print(f"  raw size: {len(wmdp)}")

wmdp_df = pd.DataFrame({
    "text": [ex["question"] for ex in wmdp],
    "source": "wmdp_bio_original",
    "label": 1,
})
wmdp_df = wmdp_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
wmdp_df["source_id"] = wmdp_df.index  # used later to keep rephrasings paired
print(f"  after dedup: {len(wmdp_df)}")

## MCQ -> prose rephrasing (deterministic)

In [ ]:
# Deterministic regex-based transformation. Returns None for stems that
# don't match any pattern (we skip rather than force a bad rephrasing).

def mcq_to_prose(stem: str):
    s = stem.strip()
    s = re.sub(
        r"\s*(choose all that apply|select the best answer|select one)\.?\s*$",
        "", s, flags=re.IGNORECASE,
    )

    m = re.match(
        r"^which of the following statements about\s+(.+?)\s+is (?:true|correct|accurate)\??$",
        s, flags=re.IGNORECASE)
    if m:
        return f"What is true about {m.group(1)}?"

    m = re.match(
        r"^which of the following best (describes|explains|characterizes)\s+(.+?)\??$",
        s, flags=re.IGNORECASE)
    if m:
        verb, obj = m.groups()
        return f"What best {verb} {obj}?"

    m = re.match(
        r"^which of the following\s+(.+?)\s+(is|are|would|could|will|can|has|have|does|do)\s+(.+?)\??$",
        s, flags=re.IGNORECASE)
    if m:
        noun, verb, rest = m.groups()
        if verb.lower() in ("is", "are"):
            return f"What {noun} {rest}?"
        return f"What {noun} {verb} {rest}?"

    m = re.match(r"^which of the following is\s+(.+?)\??$", s, flags=re.IGNORECASE)
    if m:
        return f"What is {m.group(1)}?"

    m = re.match(r"^which of the following\s+(.+?)\??$", s, flags=re.IGNORECASE)
    if m:
        return f"What {m.group(1)}?"

    return None


rephrased_rows = []
for _, row in wmdp_df.iterrows():
    prose = mcq_to_prose(row["text"])
    if prose is not None and prose.lower() != row["text"].lower():
        rephrased_rows.append({
            "text": prose,
            "source": "wmdp_bio_rephrased",
            "label": 1,
            "source_id": row["source_id"],
        })
rephrased_df = pd.DataFrame(rephrased_rows)
print(f"  WMDP-bio rephrased: {len(rephrased_df)} (from {len(wmdp_df)} candidates)")

if len(rephrased_df) > 200:
    rephrased_df = rephrased_df.sample(n=200, random_state=SEED).reset_index(drop=True)
    print(f"  Sampled down to {len(rephrased_df)}")

## Negative bucket — MMLU biology (same format, same domain)

In [ ]:
# THE critical hard negative bucket. Forces the model to learn biosecurity
# risk content rather than "is this a biology MCQ?"

print("\nLoading MMLU biology subsets...")
mmlu_bio_dfs = []
for subset in ["college_biology", "high_school_biology"]:
    for split in ["test", "validation", "dev"]:
        try:
            ds = load_dataset("cais/mmlu", subset, split=split)
            mmlu_bio_dfs.append(pd.DataFrame({
                "text": [ex["question"].strip() for ex in ds],
                "source": "mmlu_bio",
                "label": 0,
            }))
        except Exception as e:
            print(f"  skipped {subset}/{split}: {e}")
mmlu_bio_df = pd.concat(mmlu_bio_dfs, ignore_index=True).drop_duplicates("text")
print(f"  MMLU biology: {len(mmlu_bio_df)}")

## Negative bucket — MMLU non-bio technical (same format, diff domain)

In [ ]:
# Controls for "academic MCQ format = refuse" shortcut.

print("Loading MMLU non-bio technical subsets...")
mmlu_other_subsets = [
    "college_chemistry", "high_school_chemistry",
    "college_physics", "high_school_physics",
    "college_medicine", "professional_medicine", "clinical_knowledge",
]
mmlu_other_dfs = []
for subset in mmlu_other_subsets:
    for split in ["test", "validation", "dev"]:
        try:
            ds = load_dataset("cais/mmlu", subset, split=split)
            mmlu_other_dfs.append(pd.DataFrame({
                "text": [ex["question"].strip() for ex in ds],
                "source": "mmlu_other",
                "label": 0,
            }))
        except Exception as e:
            print(f"  skipped {subset}/{split}: {e}")
mmlu_other_df = pd.concat(mmlu_other_dfs, ignore_index=True).drop_duplicates("text")
print(f"  MMLU non-bio: {len(mmlu_other_df)}")

## Negative bucket — Alpaca (diff format, diff domain)

In [ ]:
# General instruction-following traffic baseline. We keep only instructions
# without an 'input' field, so prompts are comparable to the others.

print("Loading Alpaca...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca_df = pd.DataFrame([
    {"text": ex["instruction"].strip(), "source": "alpaca", "label": 0}
    for ex in alpaca
    if not ex.get("input", "").strip()
])
alpaca_df = alpaca_df.drop_duplicates("text")
print(f"  Alpaca (instructions-only): {len(alpaca_df)}")

## Negative bucket — Adjacent dual-use (the hardest slice)

In [ ]:
# Expert-authored questions topically adjacent to biosecurity but clearly
# benign: virology, medical genetics, anatomy (all MMLU) plus a keyword-
# filtered slice of PubMedQA for natural-prose biomedical research questions.

print("\nLoading adjacent dual-use sources...")
dual_use_dfs = []

# MMLU virology + medical_genetics + anatomy.
for subset in ["virology", "medical_genetics", "anatomy"]:
    for split in ["test", "validation", "dev"]:
        try:
            ds = load_dataset("cais/mmlu", subset, split=split)
            dual_use_dfs.append(pd.DataFrame({
                "text": [ex["question"].strip() for ex in ds],
                "source": "dual_use",
                "label": 0,
            }))
        except Exception as e:
            print(f"  skipped {subset}/{split}: {e}")

# PubMedQA filtered for biosecurity-adjacent topics.
try:
    pmqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    keywords = [
        "virus", "viral", "vaccine", "immune", "immunity", "pathogen",
        "infection", "epidemic", "pandemic", "antibiotic", "resistance",
        "CRISPR", "gene editing", "genetic engineering", "biosafety",
        "gain-of-function", "pathogenicity", "transmission",
    ]
    kw_pattern = re.compile("|".join(keywords), flags=re.IGNORECASE)
    pmqa_filtered = [
        {"text": ex["question"].strip(), "source": "dual_use", "label": 0}
        for ex in pmqa
        if kw_pattern.search(ex["question"])
    ]
    dual_use_dfs.append(pd.DataFrame(pmqa_filtered))
    print(f"  PubMedQA filtered: {len(pmqa_filtered)} questions matched keywords")
except Exception as e:
    print(f"  PubMedQA load failed: {e}")

dual_use_df = pd.concat(dual_use_dfs, ignore_index=True).drop_duplicates("text")
print(f"  Dual-use total: {len(dual_use_df)}")

## Cap negative buckets to target sizes

In [ ]:
# Mix matters more than absolute size. Skewed toward HARD buckets (bio + dual-use)
# relative to easy (Alpaca).

def cap_sample(df, n, seed=SEED):
    if len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

mmlu_bio_df   = cap_sample(mmlu_bio_df, 2000)
mmlu_other_df = cap_sample(mmlu_other_df, 1000)
alpaca_df     = cap_sample(alpaca_df, 3000)
dual_use_df   = cap_sample(dual_use_df, 500)

print("Final bucket sizes:")
print(f"  wmdp_bio_original:  {len(wmdp_df)}")
print(f"  wmdp_bio_rephrased: {len(rephrased_df)}")
print(f"  mmlu_bio:           {len(mmlu_bio_df)}")
print(f"  mmlu_other:         {len(mmlu_other_df)}")
print(f"  alpaca:             {len(alpaca_df)}")
print(f"  dual_use:           {len(dual_use_df)}")

## Topic-cluster WMDP positives

In [ ]:
# ============================================================================
# CELL 9: Topic-cluster WMDP positives (KMeans)
# ============================================================================
# KMeans forces balanced cluster sizes — agglomerative with average linkage
# produced one 70%-of-data mega-cluster, which collapsed topic-holdout into
# random split for most of WMDP. KMeans isn't a perfect clustering for text
# embeddings (it assumes spherical Euclidean clusters), but for the purpose
# of creating topic-separated splits, balanced clusters are what we need.

print("\nTopic-clustering WMDP positives (KMeans)...")
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

encoder = SentenceTransformer("all-MiniLM-L6-v2")
wmdp_embeddings = encoder.encode(
    wmdp_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
# L2-normalize so KMeans Euclidean distance behaves like cosine distance.
wmdp_embeddings_norm = normalize(wmdp_embeddings, norm="l2")

N_CLUSTERS = 30
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,
    random_state=SEED,
)
wmdp_df["cluster"] = kmeans.fit_predict(wmdp_embeddings_norm)

cluster_sizes = Counter(wmdp_df["cluster"])
print(f"  Cluster size distribution (top 10): {cluster_sizes.most_common(10)}")
print(f"  Cluster size distribution (bottom 5): {cluster_sizes.most_common()[-5:]}")
print(f"  Singletons: {sum(1 for s in cluster_sizes.values() if s == 1)} / {N_CLUSTERS}")
print(f"  Mean size: {np.mean(list(cluster_sizes.values())):.1f}")
print(f"  Min/Max:   {min(cluster_sizes.values())} / {max(cluster_sizes.values())}")

## Inspect clusters (optional sanity check)

In [ ]:
# Print 3 random questions per cluster. Useful for verifying topical coherence.
# Long clusters should look like coherent groupings; singletons are tail noise.

print("=" * 70)
print("CLUSTER INSPECTION — 3 random samples per cluster (largest first)")
print("=" * 70)
for cid in sorted(cluster_sizes.keys(), key=lambda c: -cluster_sizes[c]):
    size = cluster_sizes[cid]
    samples = wmdp_df[wmdp_df["cluster"] == cid].sample(
        n=min(3, size), random_state=SEED
    )
    print(f"\n--- Cluster {cid} (n={size}) ---")
    for _, row in samples.iterrows():
        text = row["text"][:200].replace("\n", " ")
        suffix = "..." if len(row["text"]) > 200 else ""
        print(f"  - {text}{suffix}")

## Rebalanced cluster-based split for WMDP positives

In [ ]:
# Pick clusters whose cumulative size hits ~20% of WMDP examples, NOT 20% of
# clusters. Otherwise heavy-tailed cluster distribution dumps tiny test set.
# (Naive 20%-of-clusters approach gave us 25 test positives — unusable.)

n_total = len(wmdp_df)
target_test_size = int(0.2 * n_total)

rng = np.random.RandomState(SEED)
cluster_list = sorted(cluster_sizes.items(), key=lambda x: x[1])  # small first
rng.shuffle(cluster_list)

test_clusters = set()
test_count = 0
for cid, size in cluster_list:
    if test_count >= target_test_size:
        break
    if test_count + size > 1.4 * target_test_size and test_count > 0.6 * target_test_size:
        continue
    test_clusters.add(cid)
    test_count += size

print(f"\nREBALANCED TEST SPLIT")
print(f"  Target: {target_test_size} ({n_total} total WMDP)")
print(f"  Actual: {test_count}")
print(f"  Test clusters: {len(test_clusters)} / {len(cluster_sizes)}")

wmdp_df["split"] = wmdp_df["cluster"].apply(
    lambda c: "test" if c in test_clusters else "train"
)
print(f"  WMDP split: {Counter(wmdp_df['split'])}")

# Pair rephrased versions to the same split as their source question.
source_to_split = dict(zip(wmdp_df["source_id"], wmdp_df["split"]))
rephrased_df["split"] = rephrased_df["source_id"].map(source_to_split)
print(f"  Rephrased split: {Counter(rephrased_df['split'])}")

## Per-bucket stratified split for negatives

In [ ]:
# Each negative bucket independently split 80/20 so test set has enough
# examples per bucket to compute meaningful per-slice recall.

def split_bucket(df, test_size=0.2, seed=SEED):
    train, test = train_test_split(df, test_size=test_size, random_state=seed)
    train = train.copy(); train["split"] = "train"
    test  = test.copy();  test["split"]  = "test"
    return pd.concat([train, test], ignore_index=True)

mmlu_bio_df   = split_bucket(mmlu_bio_df)
mmlu_other_df = split_bucket(mmlu_other_df)
alpaca_df     = split_bucket(alpaca_df)
dual_use_df   = split_bucket(dual_use_df)

## Assemble + save

In [ ]:
wmdp_out = wmdp_df.drop(columns=["cluster", "source_id"])
reph_out = rephrased_df.drop(columns=["source_id"])

full_df = pd.concat(
    [wmdp_out, reph_out, mmlu_bio_df, mmlu_other_df, alpaca_df, dual_use_df],
    ignore_index=True,
)

# Defensive cross-bucket dedup (e.g. an Alpaca instruction matching an MMLU stem).
before = len(full_df)
full_df = full_df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print(f"\nFinal dataset: {len(full_df)} rows ({before - len(full_df)} dropped as duplicates)")

print("\nSource x Split breakdown:")
print(pd.crosstab(full_df["source"], full_df["split"]))

print("\nLabel balance per split:")
print(full_df.groupby("split")["label"].value_counts())

full_df.to_parquet(OUT_DIR / "dataset.parquet", index=False)
full_df.to_csv(OUT_DIR / "dataset.csv", index=False)
full_df[full_df["split"] == "train"].to_parquet(OUT_DIR / "train.parquet", index=False)
full_df[full_df["split"] == "test"].to_parquet(OUT_DIR / "test.parquet", index=False)
print(f"\nSaved to {OUT_DIR}/")